##Step 4: Train the Customer Engagement Probability Model

In customer lifetime value calculations, we are often projecting far into the future to determine the return we might expect from a given customer or household. Inherent in these projections is an assumption that the customer will remain engaged until that point in time.  By recognizing that customer retention degrades over time, we can estimate where in a declining distribution a given customer resides and estimate a probability that the customer will stick around until the period into which we are projecting.  This logic is captured in what is known as the Beta-Geometric/Negative Binomial Distribution or BetaGeo model.  (You can read the details about this model [here](http://brucehardie.com/papers/018/fader_et_al_mksc_05.pdf).)

In [0]:
%pip install pymc-marketing==0.15.0
dbutils.library.restartPython()

In [0]:
from pymc_marketing import clv
from pymc_marketing.clv.models.beta_geo import BetaGeoModel

import pandas as pd
import numpy as np
#from datetime import timedelta

# import btyd
# from btyd.fitters.beta_geo_fitter import BetaGeoFitter
# from btyd import GammaGammaFitter

# from btyd.plotting import plot_calibration_purchases_vs_holdout_purchases
# from btyd.plotting import plot_probability_alive_matrix
# from btyd.plotting import plot_frequency_recency_matrix


# import matplotlib.pyplot as plt

import pyspark.sql.functions as fn
from pyspark.sql.types import *

# import mlflow.pyfunc
# import mlflow

Using the btyd library, we can setup such a model using either the fitters or models API.  We will use the fitters API as it appears to provide more robust functionality during evaluation and deployment:

In [0]:
filtered_cal = spark.read.format('delta').load('/tmp/clv/preprocessed')
# load spark dataframe to pandas dataframe
input_pd = filtered_cal.toPandas()
# renaming the columns to match the model requirements
input_pd.rename(columns={'customerid': 'customer_id', 'frequency_cal' : 'frequency', 'recency_cal' : 'recency', 'T_cal' : 'T'}, inplace=True)
#grouping and resetting index to help the model converge during training
bg_training_data = input_pd.groupby(["customer_id", "frequency", "recency", "T"]).size().reset_index()

# fit a model
bgf_engagement = BetaGeoModel(bg_training_data)
bgf_engagement.fit()

With our model now fit, let's make some predictions for the holdout period. We use the *conditional_expected_number_of_purchases_up_to_time* method to make this prediction.  We'll grab the actuals for that same period to enable comparison in a subsequent step:

In [0]:
# get actual frequency during holdout period
frequency_holdout_actual = input_pd['frequency_holdout']
# get predicted frequency during holdout period
frequency_holdout_predicted = bgf_engagement.expected_purchases(future_t=input_pd['duration_holdout'])

In [0]:
# flatten the xarray DataArray to a Pandas dataframe
frequency_holdout_predicted_pd = frequency_holdout_predicted.to_dataframe(name='predicted_pd').reset_index()


 With actual and predicted values in hand, we can calculate some standard evaluation metrics.  Let's wrap those calculations in a function call to make evaluation easier in future steps:
 we can calculate the RMSE for our newly trained model:

In [0]:
# define function to enable different evaluation metrics
def score_model(actuals, predicted, metric='mse'):
  metric = metric.lower() # make sure metric name is lower case
  predicted['mean'] = predicted['predicted_pd'].mean()
  
  # Mean Squared Error and Root Mean Squared Error
  if metric=='mse' or metric=='rmse':
    val = np.sum(np.square(actuals-predicted['mean']))/actuals.shape[0]
    if metric=='rmse':
        val = np.sqrt(val)
  elif metric=='mae': # Mean Absolute Error
    val = np.sum(np.abs(actuals-predicted['mean']))/actuals.shape[0]
  else:
    val = None
  
  return val

# calculate mse for predictions relative to holdout
mse = score_model(frequency_holdout_actual, frequency_holdout_predicted_pd, 'rmse')
print('RMSE: {0}'.format(mse))

While important for comparing models, the RMSE metric is a bit more challenging to interpret in terms of the overall goodness of fit of any individual model.  To provide more insight into how well our model fits our data, let's visualize the relationships between some actual and predicted values.

To get started, we can examine how purchase frequencies in the calibration period relates to actual (frequency_holdout) and predicted (model_predictions) frequencies in the holdout period:

In [0]:
#importing plotting from the lifetimes library
def plot_calibration_purchases_vs_holdout_purchases(
    model, calibration_holdout_matrix, kind="frequency_cal", n=7, **kwargs
):
    """
    Plot calibration purchases vs holdout.

    This currently relies too much on the lifetimes.util calibration_and_holdout_data function.

    Parameters
    ----------
    model: lifetimes model
        A fitted lifetimes model.
    calibration_holdout_matrix: pandas DataFrame
        DataFrame from calibration_and_holdout_data function.
    kind: str, optional
        x-axis :"frequency_cal". Purchases in calibration period,
                 "recency_cal". Age of customer at last purchase,
                 "T_cal". Age of customer at the end of calibration period,
                 "time_since_last_purchase". Time since user made last purchase
    n: int, optional
        Number of ticks on the x axis
    Returns
    -------
    axes: matplotlib.AxesSubplot

    """
    from matplotlib import pyplot as plt

    x_labels = {
        "frequency_cal": "Purchases in calibration period",
        "recency_cal": "Age of customer at last purchase",
        "T_cal": "Age of customer at the end of calibration period",
        "time_since_last_purchase": "Time since user made last purchase",
    }
    summary = calibration_holdout_matrix.copy()
    duration_holdout = summary.iloc[0]["duration_holdout"]

    summary["model_predictions"] = model.conditional_expected_number_of_purchases_up_to_time(
            duration_holdout, summary["frequency_cal"], summary["recency_cal"], summary["T_cal"])

    if kind == "time_since_last_purchase":
        summary["time_since_last_purchase"] = summary["T_cal"] - summary["recency_cal"]
        ax = (
            summary.groupby(["time_since_last_purchase"])[["frequency_holdout", "model_predictions"]]
            .mean()
            .iloc[:n]
            .plot(**kwargs)
        )
    else:
        ax = summary.groupby(kind)[["frequency_holdout", "model_predictions"]].mean().iloc[:n].plot(**kwargs)

    plt.title("Actual Purchases in Holdout Period vs Predicted Purchases")
    plt.xlabel(x_labels[kind])
    plt.ylabel("Average of Purchases in Holdout Period")
    plt.legend()

    return ax

In [0]:
plot_calibration_purchases_vs_holdout_purchases(
  bgf_engagement, 
  input_pd, 
  n=90, 
  **{'figsize':(8,8)}
  )